<a href="https://colab.research.google.com/github/gyeongee/TIL/blob/main/20_%EB%89%B4%EC%8A%A4%EA%B7%B8%EB%A3%B9_%EB%B6%84%EB%A5%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
print(f'target 클래스의 값과 분포도:{pd.Series(news_data.target).value_counts().sort_index()}')
print(f'target 클래스의 이름들\n{news_data.target_names}')

NameError: name 'news_data' is not defined

In [ ]:
print(news_data.data[0])

In [3]:
from sklearn.datasets import fetch_20newsgroups

# subset="train"으로 학습용 데이터만 추출
# remove 로 내용만 추출
train_news = fetch_20newsgroups(subset='train',remove=('header','footers','quotes'),
                               random_state=156)

X_train=train_news.data
y_train=train_news.target

# subset="test"으로 테스트 데이터만 추출
test_news = fetch_20newsgroups(subset='test',remove=('header','footers','quotes'),
                               random_state=156)

X_test = test_news.data
y_test=test_news.target
print(f'학습 데이터 크기 :{len(train_news.data)}, 테스트 데이터 크기:{len(test_news.data)}')

학습 데이터 크기 :11314, 테스트 데이터 크기:7532


In [4]:
# CountVectorizer를 이용해 학습 데이터의 텍스트를 피처 벡터화
from sklearn.feature_extraction.text import CountVectorizer

# Count Vectorization으로 피처 벡터화 변환 수행
cnt_vect = CountVectorizer()
X_train_cnt_vect=cnt_vect.fit_transform(X_train)

# 학습 데이터로 fit()된 CountVectorizer를 이용해 테스트 데이터를 피처 벡터화 변환 수행
X_test_cnt_vect = cnt_vect.transform(X_test)

print(f'학습 데이터 CountVectorizer:{X_train_cnt_vect.shape},\n'
      f'테스트 데이터 CountVectorizer:{X_test_cnt_vect.shape}')

학습 데이터 CountVectorizer:(11314, 99238),
테스트 데이터 CountVectorizer:(7532, 99238)


In [5]:
# 로지스틱 회귀로 뉴스 그룹 분류
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# LogisticRegression을 이용하여 학습/예측/평가 수행
lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_cnt_vect,y_train)
pred=lr_clf.predict(X_test_cnt_vect)
print(f'CountVectorized Logistic Regression의 예측 정확도는 {accuracy_score(y_test,pred):.3f}')

CountVectorized Logistic Regression의 예측 정확도는 0.759


In [6]:
# Count 기반에서 TF-IDF 기반으로 벡터화
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF 벡터화를 적용해 학습 데이터 세트와 테스트 데이터 세트 변환
tfidf_vect = TfidfVectorizer()
X_train_tfidf_vect=tfidf_vect.fit_transform(X_train)
X_test_tfidf_vect = tfidf_vect.transform(X_test)

#LogisticRegression을 이용해 학습/예측/평가 수행
lr_clf=LogisticRegression(solver="liblinear")
lr_clf.fit(X_train_tfidf_vect,y_train)
pred = lr_clf.predict(X_test_tfidf_vect)
print(f'TF-IDF Logistic Regression의 예측 정확도는 {accuracy_score(y_test,pred):.3f}')

TF-IDF Logistic Regression의 예측 정확도는 0.780


TF-IDF가 단순 카운트 기반보다 훨씬 높은 예측 정확도를 제공합니다.

일반적으로 문서 내에 텍스트가 많고 많은 문서를 가지는 텍스트 분석에서 카운트 벡터화보다는 TF-IDF 벡터화가 좋은 예측 결과를 도출합니다.

### 성능 향상을 위해 전처리 추가

- ngram (1,1) : 1-gram이고, 단어 하나씩 추출합니다.
- ngram(1,2): 2-gram이고, 단어 두 개씩 추출합니다.

In [7]:
# stop_words 필터링을 추가하고 ngram을 기본 (1,1)에서 (1,2)로 변경해 피처 벡터화 적용
tfidf_vect = TfidfVectorizer(stop_words='english',ngram_range=(1,2),max_df=300)
X_train_tfidf_vect=tfidf_vect.fit_transform(X_train)
X_test_tfidf_vect = tfidf_vect.transform(X_test)

#LogisticRegression을 이용해 학습/예측/평가 수행
lr_clf=LogisticRegression(solver="liblinear")
lr_clf.fit(X_train_tfidf_vect,y_train)
pred = lr_clf.predict(X_test_tfidf_vect)
print(f'TF-IDF Logistic Regression의 예측 정확도는 {accuracy_score(y_test,pred):.3f}')

TF-IDF Logistic Regression의 예측 정확도는 0.773


In [12]:
#GridSearchCV를 이용해 로지스틱 회귀 하이퍼파라미터 최적화
from sklearn.model_selection import GridSearchCV

# 최적 C값 도출 튜닝 수행. CV는 3폴드 세트로 설정
params={'C':[0.001,0.1,1,5,10]}
grid_cv_lr=GridSearchCV(lr_clf,param_grid=params,cv=3,scoring='accuracy',verbose=1)
grid_cv_lr.fit(X_train_tfidf_vect,y_train)
print(f'Logistic Regression bset C parameter:{grid_cv_lr.best_params_}')

#최적 C값으로 학습된 grid_cv로 예측 및 정확도 평가
pred = grid_cv_lr.predict(X_test_tfidf_vect)
print(f'TF-IDF Vectorized Logistic Regression의 예측 정확도는 {accuracy_score(y_test,pred):.3f}')

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Logistic Regression bset C parameter:{'C': 10}
TF-IDF Vectorized Logistic Regression의 예측 정확도는 0.805


### 사이킷런 파이프라인(Pipeline) 사용 및 GridSearchCV와의 결합

In [14]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([('tfidf_vect',TfidfVectorizer(stop_words='english')),
                     ('lr_clf',LogisticRegression(random_state=156))])


In [16]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([('tfidf_vect',TfidfVectorizer(stop_words='english',ngram_range=(1,2),max_df=300)),
                     ('lr_clf',LogisticRegression(solver='liblinear',C=10))])

# 별도의 TfidfVectorizer 객체의 fit(),transform()과 LogisticRegression의 fit(),predict() 필요없음
#pipeline의 fit()과 predict()만으로 한꺼번에 피처 벡터화와 ML 학습/예측이 가능
pipeline.fit(X_train,y_train)
pred=pipeline.predict(X_test)
print(f'Pipeline을 통한 Logistic Regression의 예측 정확도는 {accuracy_score(y_test,pred):.3f}')

Pipeline을 통한 Logistic Regression의 예측 정확도는 0.805
